# 3-Image OpenAI to HapticGen All-in-One (Colab)

This notebook runs the full validation loop **inside Colab**:

- clone/update this repo
- install repo and HapticGen dependencies
- launch a Gradio UI in Colab
- upload exactly 3 ordered images
- call OpenAI Responses API to generate structured JSON
- save `three_image_request.json`, `openai_response.json`, and `hapticgen_prompt.txt`
- send the generated JSON/prompt into HapticGen
- render `generated.wav` and `generated_waveform.png`

It also includes a second UI path for **existing `openai_response.json` -> HapticGen**.


## Before You Run

1. In Colab, add a secret named `OPENAI_API_KEY`, or set it manually in a later code cell.
2. Use a GPU runtime if possible.
3. Run the cells top to bottom, then open the Gradio link from the final cell.


In [1]:
# Setup: clone/update repo and install dependencies
from pathlib import Path
import tempfile
import subprocess
import sys

REPO_URL = 'https://github.com/cindy-77jiayi/thesis_hapticAE.git'
HAPTICGEN_REPO_URL = 'https://github.com/HapticGen/HapticGen.git'
BRANCH = 'codex/hapticgen-ui-validation'
PROJECT_ROOT = Path('/content/thesis_hapticAE')
HAPTICGEN_ROOT = Path('/content/HapticGen')

if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=False)

if not (HAPTICGEN_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', HAPTICGEN_REPO_URL, str(HAPTICGEN_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(HAPTICGEN_ROOT), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(HAPTICGEN_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

hapticgen_requirements = HAPTICGEN_ROOT / 'requirements.txt'
filtered_lines = []
skip_prefixes = ('torch==', 'torchaudio>=', 'gradio')
for raw_line in hapticgen_requirements.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    if line.startswith(skip_prefixes):
        continue
    filtered_lines.append(line)
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as handle:
    handle.write('\n'.join(filtered_lines))
    filtered_path = Path(handle.name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_path)], check=True)
try:
    import xformers  # noqa: F401
    print('xformers already available.')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xformers'], check=True)
    print('Installed xformers for current Colab runtime.')
print('Installed Colab-compatible HapticGen runtime dependencies.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(HAPTICGEN_ROOT)], check=True)
print('Installed local HapticGen package in editable mode.')

print('Project ready at:', PROJECT_ROOT)


xformers already available.
Installed Colab-compatible HapticGen runtime dependencies.
Installed local HapticGen package in editable mode.
Project ready at: /content/thesis_hapticAE


In [2]:
# Imports, environment setup, and shared helpers
from __future__ import annotations

import json
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import gradio as gr
import matplotlib.pyplot as plt
import torch
from huggingface_hub import snapshot_download
from PIL import Image
from IPython.display import Audio, display

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not os.getenv('OPENAI_API_KEY') and userdata is not None:
    try:
        secret_key = userdata.get('OPENAI_API_KEY')
        if secret_key:
            os.environ['OPENAI_API_KEY'] = secret_key
    except Exception:
        pass

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(HAPTICGEN_ROOT) not in sys.path:
    sys.path.insert(0, str(HAPTICGEN_ROOT))

from audiocraft.data.audio import audio_write
from audiocraft.models import AudioGen
from src.llm_hapticgen.openai_hapticgen import DEFAULT_HAPTICGEN_OPENAI_MODEL, OpenAIHapticGenPromptClient

DEFAULT_MODEL_ID = 'HapticGen/HapticGen-Weights'
OUTPUT_ROOT = Path('/content/outputs/hapticgen_colab_ui')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DURATION_SECONDS = 5.0

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(
        'OPENAI_API_KEY is not set. Add it as a Colab secret named OPENAI_API_KEY, '
        'or set os.environ["OPENAI_API_KEY"] manually before continuing.'
    )

print('Device:', DEVICE)
print('Output root:', OUTPUT_ROOT)
print('OpenAI model default:', DEFAULT_HAPTICGEN_OPENAI_MODEL)


Device: cuda
Output root: /content/outputs/hapticgen_colab_ui
OpenAI model default: gpt-4.1-mini


In [3]:
# HapticGen loading and artifact helpers
MODEL_CACHE: dict[str, Any] = {}

def load_hapticgen_model(model_id: str = DEFAULT_MODEL_ID, device: str | None = None):
    cache_key = f'{model_id}::{device or "auto"}'
    if cache_key in MODEL_CACHE:
        return MODEL_CACHE[cache_key]
    try:
        model = AudioGen.get_pretrained(model_id, device=device)
        loaded_from = model_id
    except Exception:
        snapshot_dir = snapshot_download(repo_id=model_id)
        model = AudioGen.get_pretrained(snapshot_dir, device=device)
        loaded_from = snapshot_dir
    model.set_generation_params(duration=DURATION_SECONDS)
    MODEL_CACHE[cache_key] = (model, loaded_from)
    return model, loaded_from

def create_run_dir(base_dir: Path = OUTPUT_ROOT) -> Path:
    base_dir.mkdir(parents=True, exist_ok=True)
    run_dir = base_dir / datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix = 1
    while run_dir.exists():
        run_dir = base_dir / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{suffix:02d}"
        suffix += 1
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

def save_waveform_plot(wav_tensor: torch.Tensor, sample_rate: int, output_path: Path) -> Path:
    waveform = wav_tensor.detach().cpu().squeeze().numpy()
    times = [idx / sample_rate for idx in range(len(waveform))]
    fig = plt.figure(figsize=(12, 3))
    ax = fig.add_subplot(111)
    ax.plot(times, waveform, linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_title('HapticGen Output Waveform')
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    return output_path

def copy_uploaded_images(image_paths: list[str], run_dir: Path) -> list[str]:
    images_dir = run_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for index, image_path in enumerate(image_paths, start=1):
        source = Path(image_path).resolve()
        target = images_dir / f'image_{index:02d}{source.suffix.lower()}'
        shutil.copy2(source, target)
        copied.append(str(target))
    return copied

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

def generate_haptic_outputs(prompt: str, run_dir: Path, model_id: str = DEFAULT_MODEL_ID) -> dict[str, str]:
    model, loaded_from = load_hapticgen_model(model_id=model_id, device=DEVICE)
    generated = model.generate([prompt])
    wav_tensor = generated[0].detach().cpu()

    wav_stem = run_dir / 'generated'
    wav_file = audio_write(
        wav_stem,
        wav_tensor,
        model.sample_rate,
        format='wav',
        normalize=False,
    )
    waveform_file = save_waveform_plot(wav_tensor, model.sample_rate, run_dir / 'generated_waveform.png')
    metadata = {
        'prompt': prompt,
        'model_id': model_id,
        'loaded_from': loaded_from,
        'duration': DURATION_SECONDS,
        'sample_rate': model.sample_rate,
        'wav_path': str(wav_file),
        'waveform_image_path': str(waveform_file),
    }
    write_json(run_dir / 'generation_metadata.json', metadata)
    return {
        'wav_path': str(wav_file),
        'waveform_image_path': str(waveform_file),
        'metadata_path': str((run_dir / 'generation_metadata.json').resolve()),
        'loaded_from': str(loaded_from),
    }

def run_end_to_end(image_1: str | None, image_2: str | None, image_3: str | None, notes: str, openai_model: str, hapticgen_model_id: str):
    image_paths = [image_1, image_2, image_3]
    if any(path in (None, '') for path in image_paths):
        raise gr.Error('Please upload exactly 3 images before generating.')
    run_dir = create_run_dir()
    copied_images = copy_uploaded_images([str(path) for path in image_paths], run_dir)
    request_payload = {
        'image_paths': copied_images,
        'notes': notes or '',
        'openai_model': openai_model,
    }
    write_json(run_dir / 'three_image_request.json', request_payload)

    client = OpenAIHapticGenPromptClient(model=openai_model, template_path=PROJECT_ROOT / 'llm' / 'hapticgen_prompt_template.md')
    result = client.infer_prompt(image_paths=copied_images, notes=notes or '')
    response_payload = {
        'visual_summary': result['visual_summary'],
        'haptic_prompt': result['haptic_prompt'],
        'negative_prompt': result['negative_prompt'],
        'rationale': result['rationale'],
        'response_text': result['response_text'],
    }
    response_json = run_dir / 'openai_response.json'
    write_json(response_json, response_payload)
    prompt_txt = run_dir / 'hapticgen_prompt.txt'
    prompt_txt.write_text(result['haptic_prompt'].strip(), encoding='utf-8')

    generation = generate_haptic_outputs(result['haptic_prompt'], run_dir, model_id=hapticgen_model_id)
    return (
        result['visual_summary'],
        result['haptic_prompt'],
        result['negative_prompt'],
        result['rationale'],
        str(run_dir),
        str(response_json),
        str(prompt_txt),
        generation['wav_path'],
        generation['waveform_image_path'],
        generation['metadata_path'],
        f"Loaded HapticGen from: {generation['loaded_from']}",
    )

def run_from_response_json(response_json_path: str | None, hapticgen_model_id: str):
    if not response_json_path:
        raise gr.Error('Upload an openai_response.json file first.')
    source = Path(response_json_path).resolve()
    payload = json.loads(source.read_text(encoding='utf-8'))
    prompt = str(payload.get('haptic_prompt', '')).strip()
    if not prompt:
        raise gr.Error('The uploaded JSON does not contain a non-empty haptic_prompt.')
    run_dir = create_run_dir()
    copied_json = run_dir / 'openai_response.json'
    shutil.copy2(source, copied_json)
    (run_dir / 'hapticgen_prompt.txt').write_text(prompt, encoding='utf-8')
    generation = generate_haptic_outputs(prompt, run_dir, model_id=hapticgen_model_id)
    return (
        prompt,
        str(run_dir),
        str(copied_json),
        generation['wav_path'],
        generation['waveform_image_path'],
        generation['metadata_path'],
        f"Loaded HapticGen from: {generation['loaded_from']}",
    )


In [4]:
# Optional warm-up: load HapticGen once before launching the UI
model, loaded_from = load_hapticgen_model(model_id=DEFAULT_MODEL_ID, device=DEVICE)
print('HapticGen loaded from:', loaded_from)
print('Sample rate:', model.sample_rate)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


compression_state_dict.bin:   0%|          | 0.00/236M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


state_dict.bin:   0%|          | 0.00/3.68G [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

HapticGen loaded from: HapticGen/HapticGen-Weights
Sample rate: 8000


In [5]:
# Build and launch the all-in-one Colab UI
with gr.Blocks(title='3-Image OpenAI to HapticGen (Colab)') as demo:
    gr.Markdown(
        '''
        # 3-Image OpenAI to HapticGen (Colab)
        This UI runs the full validation loop in Colab:
        3 images -> OpenAI JSON -> HapticGen -> wav + waveform.
        '''
    )

    with gr.Row():
        openai_model = gr.Textbox(label='OpenAI model', value=DEFAULT_HAPTICGEN_OPENAI_MODEL)
        hapticgen_model_id = gr.Textbox(label='HapticGen model', value=DEFAULT_MODEL_ID)

    with gr.Tab('3-image end-to-end'):
        with gr.Row():
            image_1 = gr.Image(label='image_1', type='filepath')
            image_2 = gr.Image(label='image_2', type='filepath')
            image_3 = gr.Image(label='image_3', type='filepath')
        notes = gr.Textbox(label='optional notes', lines=3, placeholder='Optional timing or tactile notes')
        generate_btn = gr.Button('Generate', variant='primary')

        with gr.Row():
            visual_summary = gr.Textbox(label='OpenAI summary', lines=4)
            haptic_prompt = gr.Textbox(label='HapticGen prompt', lines=4)
        with gr.Row():
            negative_prompt = gr.Textbox(label='negative prompt', lines=2)
            rationale = gr.Textbox(label='rationale', lines=4)
        run_dir = gr.Textbox(label='run directory', interactive=False)
        response_json_file = gr.File(label='openai_response.json')
        prompt_txt_file = gr.File(label='hapticgen_prompt.txt')
        audio_output = gr.Audio(label='generated wav')
        waveform_output = gr.Image(label='waveform image')
        metadata_file = gr.File(label='generation metadata')
        model_status = gr.Textbox(label='model status', interactive=False)

        generate_btn.click(
            run_end_to_end,
            inputs=[image_1, image_2, image_3, notes, openai_model, hapticgen_model_id],
            outputs=[
                visual_summary,
                haptic_prompt,
                negative_prompt,
                rationale,
                run_dir,
                response_json_file,
                prompt_txt_file,
                audio_output,
                waveform_output,
                metadata_file,
                model_status,
            ],
        )

    with gr.Tab('existing JSON -> HapticGen'):
        response_json_upload = gr.File(label='Upload openai_response.json', file_types=['.json'], type='filepath')
        generate_from_json_btn = gr.Button('Generate from JSON')
        json_prompt = gr.Textbox(label='prompt from JSON', lines=4)
        json_run_dir = gr.Textbox(label='run directory', interactive=False)
        copied_json_output = gr.File(label='copied openai_response.json')
        json_audio_output = gr.Audio(label='generated wav')
        json_waveform_output = gr.Image(label='waveform image')
        json_metadata_output = gr.File(label='generation metadata')
        json_model_status = gr.Textbox(label='model status', interactive=False)

        generate_from_json_btn.click(
            run_from_response_json,
            inputs=[response_json_upload, hapticgen_model_id],
            outputs=[
                json_prompt,
                json_run_dir,
                copied_json_output,
                json_audio_output,
                json_waveform_output,
                json_metadata_output,
                json_model_status,
            ],
        )

demo.launch(share=True, debug=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8431264e7824775dd2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8431264e7824775dd2.gradio.live


In [6]:
# Optional preview helper: point this to the newest run dir if you want inline notebook previews too
LATEST_RUN = None

if LATEST_RUN:
    latest = Path(LATEST_RUN)
    display(Audio(str(latest / 'generated.wav')))
    display(Image.open(latest / 'generated_waveform.png'))
